# Experiment 7: Sentiment Analysis on IMDB Movie Reviews Using Logistic Regression, LSTM, Vanilla Transformer, and Pretrained BERT

**Course**: Computational Linguistics and Natural Language Processing (CLNLP) Lab  
**Student Name**: Satwik Shukla  
**SAP ID**: 500123679  
**Batch**: 1 AIML | B.Tech CSE  

---

### Objectives:
1. Implement sentiment analysis on the IMDB Movie Reviews dataset using four distinct modeling paradigms:
   - Logistic Regression with TF-IDF features
   - Bidirectional Long Short-Term Memory (LSTM) network in PyTorch
   - Vanilla Transformer Encoder in PyTorch
   - Pretrained BERT-family Transformer model via HuggingFace
2. Perform comprehensive evaluation across all four models using Precision, Recall, F1-Score, and Confusion Matrix.
3. Conduct a comparative analysis of model performance, architecture trade-offs, and computational complexity.


In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import re
import numpy as np
import pandas as pd
from collections import Counter
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix
from datasets import load_dataset
from transformers import pipeline


## 1. Dataset Loading and Preprocessing
We load the Stanford IMDB Movie Reviews dataset and prepare balanced train and test splits for model training and benchmark evaluation.


In [2]:
dataset = load_dataset('stanfordnlp/imdb')
train_subset = dataset['train'].shuffle(seed=42).select(range(600))
test_subset = dataset['test'].shuffle(seed=42).select(range(200))

train_texts = [item['text'] for item in train_subset]
train_labels = np.array(train_subset['label'])
test_texts = [item['text'] for item in test_subset]
test_labels = np.array(test_subset['label'])

def clean_and_tokenize(text):
    return re.sub(r'[^a-zA-Z ]', ' ', text.lower()).split()

all_train_tokens = [tok for text in train_texts for tok in clean_and_tokenize(text)]
vocab_counts = Counter(all_train_tokens).most_common(5000)
word_to_id = {w: i + 2 for i, (w, c) in enumerate(vocab_counts)}
word_to_id['<pad>'] = 0
word_to_id['<unk>'] = 1

max_seq_length = 100

def text_to_padded_ids(text, max_len=max_seq_length):
    tokens = clean_and_tokenize(text)[:max_len]
    token_ids = [word_to_id.get(tok, 1) for tok in tokens]
    if len(token_ids) < max_len:
        token_ids += [0] * (max_len - len(token_ids))
    return token_ids

train_tensor_ids = torch.tensor([text_to_padded_ids(t) for t in train_texts], dtype=torch.long)
train_tensor_labels = torch.tensor(train_labels, dtype=torch.long)
test_tensor_ids = torch.tensor([text_to_padded_ids(t) for t in test_texts], dtype=torch.long)
test_tensor_labels = torch.tensor(test_labels, dtype=torch.long)

train_dataloader = DataLoader(TensorDataset(train_tensor_ids, train_tensor_labels), batch_size=32, shuffle=True)
test_dataloader = DataLoader(TensorDataset(test_tensor_ids, test_tensor_labels), batch_size=32, shuffle=False)

print('Dataset loaded successfully:')
print('Training set size:', len(train_texts), 'Test set size:', len(test_texts))
print('Vocabulary size:', len(word_to_id))


Dataset loaded successfully:
Training set size: 600 Test set size: 200
Vocabulary size: 5002


## 2. Model 1: Logistic Regression with TF-IDF Representation
We extract n-gram TF-IDF representations (unigrams and bigrams) and optimize the cross-entropy objective via L2-regularized Logistic Regression.


In [3]:
tfidf_vectorizer = TfidfVectorizer(max_features=2500, ngram_range=(1, 2))
X_train_tfidf = tfidf_vectorizer.fit_transform(train_texts)
X_test_tfidf = tfidf_vectorizer.transform(test_texts)

lr_model = LogisticRegression(max_iter=500, random_state=42)
lr_model.fit(X_train_tfidf, train_labels)

lr_predictions = lr_model.predict(X_test_tfidf)
lr_precision, lr_recall, lr_f1, _ = precision_recall_fscore_support(test_labels, lr_predictions, average='binary')
lr_accuracy = accuracy_score(test_labels, lr_predictions)
lr_cm = confusion_matrix(test_labels, lr_predictions)

print('=== Model 1: Logistic Regression Results ===')
print('Accuracy :', round(float(lr_accuracy), 4))
print('Precision:', round(float(lr_precision), 4))
print('Recall   :', round(float(lr_recall), 4))
print('F1-Score :', round(float(lr_f1), 4))
print('Confusion Matrix:')
print(lr_cm)


=== Model 1: Logistic Regression Results ===
Accuracy : 0.74
Precision: 0.75
Recall   : 0.6875
F1-Score : 0.7174
Confusion Matrix:
[[82 22]
 [30 66]]


## 3. Model 2: Bidirectional LSTM Neural Network
We construct a Bidirectional LSTM in PyTorch with an embedding layer, hidden recurrences processing sequences in forward and reverse directions, and a classification projection head.


In [4]:
class BidirectionalLSTMClassifier(nn.Module):
    def __init__(self, vocab_dim, embedding_dim=64, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_dim, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.classifier = nn.Linear(hidden_dim * 2, 2)
    def forward(self, input_ids):
        embedded = self.embedding(input_ids)
        lstm_out, _ = self.lstm(embedded)
        mean_pooled = torch.mean(lstm_out, dim=1)
        return self.classifier(mean_pooled)

torch.manual_seed(42)
lstm_network = BidirectionalLSTMClassifier(len(word_to_id) + 2)
lstm_criterion = nn.CrossEntropyLoss()
lstm_optimizer = torch.optim.Adam(lstm_network.parameters(), lr=0.003)

for epoch in range(4):
    lstm_network.train()
    for batch_x, batch_y in train_dataloader:
        lstm_optimizer.zero_grad()
        logits = lstm_network(batch_x)
        loss = lstm_criterion(logits, batch_y)
        loss.backward()
        lstm_optimizer.step()

lstm_network.eval()
all_lstm_preds = []
with torch.no_grad():
    for batch_x, _ in test_dataloader:
        batch_logits = lstm_network(batch_x)
        batch_preds = torch.argmax(batch_logits, dim=-1).cpu().numpy()
        all_lstm_preds.extend(batch_preds)

lstm_predictions = np.array(all_lstm_preds)
lstm_precision, lstm_recall, lstm_f1, _ = precision_recall_fscore_support(test_labels, lstm_predictions, average='binary')
lstm_accuracy = accuracy_score(test_labels, lstm_predictions)
lstm_cm = confusion_matrix(test_labels, lstm_predictions)

print('=== Model 2: Bidirectional LSTM Results ===')
print('Accuracy :', round(float(lstm_accuracy), 4))
print('Precision:', round(float(lstm_precision), 4))
print('Recall   :', round(float(lstm_recall), 4))
print('F1-Score :', round(float(lstm_f1), 4))
print('Confusion Matrix:')
print(lstm_cm)


=== Model 2: Bidirectional LSTM Results ===
Accuracy : 0.63
Precision: 0.6038
Recall   : 0.6667
F1-Score : 0.6337
Confusion Matrix:
[[62 42]
 [32 64]]


## 4. Model 3: Vanilla Transformer Encoder
We construct a Vanilla Transformer Encoder in PyTorch incorporating learned positional embeddings, multi-head self-attention mechanisms, and residual feedforward projections.


In [5]:
class VanillaTransformerClassifier(nn.Module):
    def __init__(self, vocab_dim, max_len=100, embedding_dim=64, num_heads=4, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_dim, embedding_dim, padding_idx=0)
        self.positional_embedding = nn.Embedding(max_len, embedding_dim)
        encoder_layer = nn.TransformerEncoderLayer(d_model=embedding_dim, nhead=num_heads, dim_feedforward=hidden_dim, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.classifier = nn.Linear(embedding_dim, 2)
    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape
        pos_indices = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(batch_size, -1)
        tokens_plus_positions = self.embedding(input_ids) + self.positional_embedding(pos_indices)
        encoded_representations = self.transformer_encoder(tokens_plus_positions)
        mean_pooled = torch.mean(encoded_representations, dim=1)
        return self.classifier(mean_pooled)

torch.manual_seed(42)
transformer_network = VanillaTransformerClassifier(len(word_to_id) + 2)
transformer_criterion = nn.CrossEntropyLoss()
transformer_optimizer = torch.optim.Adam(transformer_network.parameters(), lr=0.001)

for epoch in range(4):
    transformer_network.train()
    for batch_x, batch_y in train_dataloader:
        transformer_optimizer.zero_grad()
        logits = transformer_network(batch_x)
        loss = transformer_criterion(logits, batch_y)
        loss.backward()
        transformer_optimizer.step()

transformer_network.eval()
all_transformer_preds = []
with torch.no_grad():
    for batch_x, _ in test_dataloader:
        batch_logits = transformer_network(batch_x)
        batch_preds = torch.argmax(batch_logits, dim=-1).cpu().numpy()
        all_transformer_preds.extend(batch_preds)

transformer_predictions = np.array(all_transformer_preds)
trans_precision, trans_recall, trans_f1, _ = precision_recall_fscore_support(test_labels, transformer_predictions, average='binary')
trans_accuracy = accuracy_score(test_labels, transformer_predictions)
trans_cm = confusion_matrix(test_labels, transformer_predictions)

print('=== Model 3: Vanilla Transformer Results ===')
print('Accuracy :', round(float(trans_accuracy), 4))
print('Precision:', round(float(trans_precision), 4))
print('Recall   :', round(float(trans_recall), 4))
print('F1-Score :', round(float(trans_f1), 4))
print('Confusion Matrix:')
print(trans_cm)


=== Model 3: Vanilla Transformer Results ===
Accuracy : 0.55
Precision: 0.5294
Recall   : 0.5625
F1-Score : 0.5455
Confusion Matrix:
[[56 48]
 [42 54]]


## 5. Model 4: Pretrained BERT Transformer Model
We deploy a pretrained Transformer sequence classifier (`distilbert/distilbert-base-uncased-finetuned-sst-2-english`) and evaluate zero-shot transfer capabilities on the IMDB test partition.


In [6]:
sentiment_pipeline = pipeline('text-classification', model='distilbert/distilbert-base-uncased-finetuned-sst-2-english', device=-1)

bert_predictions_list = []
batch_size_bert = 32
for i in range(0, len(test_texts), batch_size_bert):
    batch_texts = [text[:512] for text in test_texts[i:i + batch_size_bert]]
    batch_outputs = sentiment_pipeline(batch_texts)
    for output in batch_outputs:
        label_int = 1 if output['label'] == 'POSITIVE' else 0
        bert_predictions_list.append(label_int)

bert_predictions = np.array(bert_predictions_list)
bert_precision, bert_recall, bert_f1, _ = precision_recall_fscore_support(test_labels, bert_predictions, average='binary')
bert_accuracy = accuracy_score(test_labels, bert_predictions)
bert_cm = confusion_matrix(test_labels, bert_predictions)

print('=== Model 4: Pretrained BERT Model Results ===')
print('Accuracy :', round(float(bert_accuracy), 4))
print('Precision:', round(float(bert_precision), 4))
print('Recall   :', round(float(bert_recall), 4))
print('F1-Score :', round(float(bert_f1), 4))
print('Confusion Matrix:')
print(bert_cm)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

=== Model 4: Pretrained BERT Model Results ===
Accuracy : 0.79
Precision: 0.77
Recall   : 0.8021
F1-Score : 0.7857
Confusion Matrix:
[[81 23]
 [19 77]]


## 6. Comparative Performance Analysis
We consolidate Precision, Recall, F1-Score, and Accuracy across all four paradigms into a unified comparative table.


In [7]:
summary_records = [
    {
        'Model Architecture': 'Logistic Regression (TF-IDF)',
        'Accuracy': round(float(lr_accuracy), 4),
        'Precision': round(float(lr_precision), 4),
        'Recall': round(float(lr_recall), 4),
        'F1-Score': round(float(lr_f1), 4)
    },
    {
        'Model Architecture': 'Bidirectional LSTM (PyTorch)',
        'Accuracy': round(float(lstm_accuracy), 4),
        'Precision': round(float(lstm_precision), 4),
        'Recall': round(float(lstm_recall), 4),
        'F1-Score': round(float(lstm_f1), 4)
    },
    {
        'Model Architecture': 'Vanilla Transformer (PyTorch)',
        'Accuracy': round(float(trans_accuracy), 4),
        'Precision': round(float(trans_precision), 4),
        'Recall': round(float(trans_recall), 4),
        'F1-Score': round(float(trans_f1), 4)
    },
    {
        'Model Architecture': 'Pretrained BERT (Transformers)',
        'Accuracy': round(float(bert_accuracy), 4),
        'Precision': round(float(bert_precision), 4),
        'Recall': round(float(bert_recall), 4),
        'F1-Score': round(float(bert_f1), 4)
    }
]

df_comparative = pd.DataFrame(summary_records)
print('=== Comprehensive Model Performance Comparison ===')
print(df_comparative)

print('=== Detailed Confusion Matrices ===')
print('1. Logistic Regression Confusion Matrix:')
print(lr_cm)
print('2. Bidirectional LSTM Confusion Matrix:')
print(lstm_cm)
print('3. Vanilla Transformer Confusion Matrix:')
print(trans_cm)
print('4. Pretrained BERT Confusion Matrix:')
print(bert_cm)


=== Comprehensive Model Performance Comparison ===
               Model Architecture  Accuracy  Precision  Recall  F1-Score
0    Logistic Regression (TF-IDF)      0.74     0.7500  0.6875    0.7174
1    Bidirectional LSTM (PyTorch)      0.63     0.6038  0.6667    0.6337
2   Vanilla Transformer (PyTorch)      0.55     0.5294  0.5625    0.5455
3  Pretrained BERT (Transformers)      0.79     0.7700  0.8021    0.7857
=== Detailed Confusion Matrices ===
1. Logistic Regression Confusion Matrix:
[[82 22]
 [30 66]]
2. Bidirectional LSTM Confusion Matrix:
[[62 42]
 [32 64]]
3. Vanilla Transformer Confusion Matrix:
[[56 48]
 [42 54]]
4. Pretrained BERT Confusion Matrix:
[[81 23]
 [19 77]]
